In [1]:
from transformers import pipeline
import time

print("Environment is ready")

Environment is ready


In [2]:
MODEL_NAME = "facebook/bart-large-mnli"

In [3]:
classifier = pipeline(
    task="zero-shot-classification",
    model=MODEL_NAME
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [4]:
candidate_labels = [
    "data engineering and data pipelines",
    "artificial intelligence and machine learning",
    "devops and deployment automation",
    "cloud infrastructure and cloud engineering",
    "cybersecurity and information security",
    "software and application development",
]

In [5]:
job_description = """
We are looking for a data engineer with experience in Python, SQL,
Apache Airflow, dbt, Spark, PostgreSQL, and building ETL pipelines
for data warehouses.
"""

In [6]:
result = classifier(
    job_description,
    candidate_labels=candidate_labels
)
print(result)

{'sequence': '\nWe are looking for a data engineer with experience in Python, SQL,\nApache Airflow, dbt, Spark, PostgreSQL, and building ETL pipelines\nfor data warehouses.\n', 'labels': ['data engineering and data pipelines', 'software and application development', 'artificial intelligence and machine learning', 'cloud infrastructure and cloud engineering', 'devops and deployment automation', 'cybersecurity and information security'], 'scores': [0.9227437376976013, 0.02448762208223343, 0.020575424656271935, 0.017761483788490295, 0.009550684131681919, 0.00488107418641448]}


In [7]:
import pandas as pd

results_df = pd.DataFrame({
    "Category": result["labels"],
    "Confidence": result["scores"]
})

results_df

,Category,Confidence
0,data engineering and data pipelines,0.922744
1,software and application development,0.024488
2,artificial intelligence and machine learning,0.020575
3,cloud infrastructure and cloud engineering,0.017761
4,devops and deployment automation,0.009551
5,cybersecurity and information security,0.004881


In [8]:
results_df = pd.DataFrame({
    "Rank": range(1, len(result["labels"]) + 1),
    "Category": result["labels"],
    "Confidence (%)": [
        round(score * 100, 2)
        for score in result["scores"]
    ]
})

predicted_category = results_df.iloc[0]["Category"]
predicted_confidence = results_df.iloc[0]["Confidence (%)"]

print("Predicted Category:", predicted_category)
print("Confidence:", predicted_confidence, "%")

results_df

Predicted Category: data engineering and data pipelines
Confidence: 92.27 %


,Rank,Category,Confidence (%)
0,1,data engineering and data pipelines,92.27
1,2,software and application development,2.45
2,3,artificial intelligence and machine learning,2.06
3,4,cloud infrastructure and cloud engineering,1.78
4,5,devops and deployment automation,0.96
5,6,cybersecurity and information security,0.49


In [9]:
job_description = """
We are looking for an engineer to design and maintain scalable data platforms
on AWS. The role includes building ETL pipelines with Python and Spark,
orchestrating workflows with Apache Airflow, managing Docker containers,
deploying services through CI/CD pipelines, and monitoring production systems.

The candidate will also work with data warehouses, cloud storage,
infrastructure automation, and system reliability.
"""

In [10]:
result = classifier(
    job_description,
    candidate_labels=candidate_labels
)

In [11]:
results_df = pd.DataFrame({
    "Rank": range(1, len(result["labels"]) + 1),
    "Category": result["labels"],
    "Confidence (%)": [
        round(score * 100, 2)
        for score in result["scores"]
    ]
})

print("Predicted Category:", results_df.iloc[0]["Category"])
print("Confidence:", results_df.iloc[0]["Confidence (%)"], "%")

results_df

Predicted Category: cloud infrastructure and cloud engineering
Confidence: 48.49 %


,Rank,Category,Confidence (%)
0,1,cloud infrastructure and cloud engineering,48.49
1,2,data engineering and data pipelines,32.49
2,3,devops and deployment automation,14.33
3,4,software and application development,2.75
4,5,artificial intelligence and machine learning,1.25
5,6,cybersecurity and information security,0.68


In [12]:
import pandas as pd

evaluation_df = pd.read_csv("data/job_posts_evaluation.csv")
evaluation_df

,job_description,expected_category
0,We need an engineer to build ETL pipelines usi...,data engineering and data pipelines
1,"Design dimensional models, maintain data wareh...",data engineering and data pipelines
2,Build machine learning models for text classif...,artificial intelligence and machine learning
3,"Develop RAG systems using embeddings, vector d...",artificial intelligence and machine learning
4,"Manage Kubernetes clusters, automate CI/CD pip...",devops and deployment automation
5,Create automated release pipelines using Docke...,devops and deployment automation
6,"Design scalable AWS infrastructure using EC2, ...",cloud infrastructure and cloud engineering
7,Build secure Azure cloud environments and mana...,cloud infrastructure and cloud engineering
8,Monitor security events using SIEM tools and i...,cybersecurity and information security
9,"Perform vulnerability assessments, threat dete...",cybersecurity and information security


In [13]:


predictions = []
confidences = []
latencies = []

for job_description in evaluation_df["job_description"]:
    start_time = time.perf_counter()

    result = classifier(
        job_description,
        candidate_labels=candidate_labels,
        multi_label=False
    )

    latency = time.perf_counter() - start_time

    predictions.append(result["labels"][0])
    confidences.append(round(result["scores"][0] * 100, 2))
    latencies.append(round(latency, 3))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [14]:
evaluation_df["predicted_category"] = predictions
evaluation_df["confidence_percent"] = confidences
evaluation_df["latency_seconds"] = latencies

evaluation_df["is_correct"] = (
    evaluation_df["expected_category"]
    == evaluation_df["predicted_category"]
)

In [15]:
evaluation_df[
    [
        "job_description",
        "expected_category",
        "predicted_category",
        "confidence_percent",
        "latency_seconds",
        "is_correct",
    ]
]

,job_description,expected_category,predicted_category,confidence_percent,latency_seconds,is_correct
0,We need an engineer to build ETL pipelines usi...,data engineering and data pipelines,data engineering and data pipelines,76.93,0.338,True
1,"Design dimensional models, maintain data wareh...",data engineering and data pipelines,data engineering and data pipelines,69.28,0.150,True
2,Build machine learning models for text classif...,artificial intelligence and machine learning,artificial intelligence and machine learning,77.80,0.113,True
3,"Develop RAG systems using embeddings, vector d...",artificial intelligence and machine learning,data engineering and data pipelines,30.38,0.114,False
4,"Manage Kubernetes clusters, automate CI/CD pip...",devops and deployment automation,devops and deployment automation,68.26,0.116,True
5,Create automated release pipelines using Docke...,devops and deployment automation,devops and deployment automation,63.71,0.123,True
6,"Design scalable AWS infrastructure using EC2, ...",cloud infrastructure and cloud engineering,cloud infrastructure and cloud engineering,90.63,0.118,True
7,Build secure Azure cloud environments and mana...,cloud infrastructure and cloud engineering,cloud infrastructure and cloud engineering,85.59,0.116,True
8,Monitor security events using SIEM tools and i...,cybersecurity and information security,cybersecurity and information security,71.86,0.117,True
9,"Perform vulnerability assessments, threat dete...",cybersecurity and information security,cybersecurity and information security,75.65,0.116,True


In [16]:
bart_summary = {
    "model": MODEL_NAME,
    "total_examples": len(evaluation_df),
    "correct_predictions": int(evaluation_df["is_correct"].sum()),
    "wrong_predictions": int((~evaluation_df["is_correct"]).sum()),
    "accuracy_percent": round(
        evaluation_df["is_correct"].mean() * 100,
        2
    ),
    "average_confidence_percent": round(
        evaluation_df["confidence_percent"].mean(),
        2
    ),
    "average_latency_seconds": round(
        evaluation_df["latency_seconds"].mean(),
        3
    ),
}

pd.DataFrame([bart_summary])

,model,total_examples,correct_predictions,wrong_predictions,accuracy_percent,average_confidence_percent,average_latency_seconds
0,facebook/bart-large-mnli,12,11,1,91.67,75.23,0.137


In [19]:
# 11 correct / 12 total
# Accuracy: 91.67%
# Preliminary result on a small evaluation set

In [17]:
bart_summary = {
    "model_name": "BART",
    "model_id": "facebook/bart-large-mnli",

    "accuracy_percent": round(
        evaluation_df["is_correct"].mean() * 100,
        2
    ),

    "average_latency_seconds": round(
        evaluation_df["latency_seconds"].mean(),
        4
    ),

    "average_confidence_percent": round(
        evaluation_df["confidence_percent"].mean(),
        2
    ),

    "cost_usd": 0.0,
    "cost_description": "Free - Local inference",

    "correct_predictions": int(
        evaluation_df["is_correct"].sum()
    ),

    "wrong_predictions": int(
        (~evaluation_df["is_correct"]).sum()
    ),

    "total_examples": int(
        len(evaluation_df)
    ),
}

bart_final_result = pd.DataFrame([bart_summary])

bart_final_result

,model_name,model_id,accuracy_percent,average_latency_seconds,average_confidence_percent,cost_usd,cost_description,correct_predictions,wrong_predictions,total_examples
0,BART,facebook/bart-large-mnli,91.67,0.1369,75.23,0.0,Free - Local inference,11,1,12


In [18]:
from pathlib import Path
import pandas as pd

RESULTS_FILE = Path(
    "data/final_result_of_comparison.csv"
)

if RESULTS_FILE.exists():
    comparison_df = pd.read_csv(RESULTS_FILE)

    comparison_df = comparison_df[
        comparison_df["model_name"] != "BART"
    ]

    comparison_df = pd.concat(
        [
            comparison_df,
            bart_final_result
        ],
        ignore_index=True
    )

else:
    comparison_df = bart_final_result.copy()

comparison_df = comparison_df.sort_values(
    by="accuracy_percent",
    ascending=False
).reset_index(drop=True)

comparison_df.to_csv(
    RESULTS_FILE,
    index=False
)

comparison_df

,model_name,model_id,accuracy_percent,average_latency_seconds,average_confidence_percent,cost_usd,cost_description,correct_predictions,wrong_predictions,total_examples
0,DeBERTa,MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli,91.67,0.2268,78.77,0.0,Free - Local inference,11,1,12
1,BART,facebook/bart-large-mnli,91.67,0.1369,75.23,0.0,Free - Local inference,11,1,12
2,ModernBERT,tasksource/ModernBERT-base-nli,83.33,0.0977,41.82,0.0,Free - Local inference,10,2,12


In [19]:
comparison_df

,model_name,model_id,accuracy_percent,average_latency_seconds,average_confidence_percent,cost_usd,cost_description,correct_predictions,wrong_predictions,total_examples
0,DeBERTa,MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli,91.67,0.2268,78.77,0.0,Free - Local inference,11,1,12
1,BART,facebook/bart-large-mnli,91.67,0.1369,75.23,0.0,Free - Local inference,11,1,12
2,ModernBERT,tasksource/ModernBERT-base-nli,83.33,0.0977,41.82,0.0,Free - Local inference,10,2,12
